# Notebook 04: Portfolio Construction

Explores the mean-variance optimization layer:
- **Efficient frontier**: risk vs. return trade-off across portfolios
- **Weight evolution heatmap**: how stock weights shift over time
- **Turnover over time**: the cost of rebalancing

In [ ]:
import sys
sys.path.append('..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.optimize import minimize

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline
print('✅ Imports successful')

## 1. Load Demo Data

In [ ]:
raw_dir = Path('../data/raw')
parquet_files = sorted(raw_dir.glob('*.parquet'))
demo_file = [f for f in parquet_files if '10tickers' in f.name] or [parquet_files[0]]
data = pd.read_parquet(demo_file[0])
tickers = data.index.get_level_values('ticker').unique().tolist()
print(f'Tickers: {tickers}')
print(f'Shape: {data.shape}')

## 2. Efficient Frontier

We sample 300 random portfolios and one mean-variance optimal portfolio (max Sharpe).

In [ ]:
close_wide = data['close'].unstack('ticker').dropna()
returns = np.log(close_wide / close_wide.shift(1)).dropna()

mu = returns.mean() * 252
cov = returns.cov() * 252
n = len(tickers)

np.random.seed(42)
n_portfolios = 300
port_returns, port_vols = [], []

for _ in range(n_portfolios):
    w = np.random.dirichlet(np.ones(n))
    port_returns.append(float(w @ mu))
    port_vols.append(float(np.sqrt(w @ cov.values @ w)))

def neg_sharpe(w):
    r = w @ mu
    v = np.sqrt(w @ cov.values @ w)
    return -r / (v + 1e-9)

result = minimize(
    neg_sharpe,
    x0=np.ones(n) / n,
    bounds=[(0, 0.3)] * n,
    constraints={'type': 'eq', 'fun': lambda w: w.sum() - 1},
    method='SLSQP',
)
opt_w = result.x
opt_ret = float(opt_w @ mu)
opt_vol = float(np.sqrt(opt_w @ cov.values @ opt_w))
opt_sharpe = opt_ret / opt_vol

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(port_vols, port_returns, c=np.array(port_returns) / np.array(port_vols),
                     cmap='viridis', alpha=0.5, s=10)
ax.scatter(opt_vol, opt_ret, marker='*', s=300, color='red',
           label=f'Max Sharpe ({opt_sharpe:.2f})')
plt.colorbar(scatter, ax=ax, label='Sharpe Ratio')
ax.set_xlabel('Annualized Volatility')
ax.set_ylabel('Annualized Return')
ax.set_title('Efficient Frontier (300 Random Portfolios)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'\nOptimal portfolio (Max Sharpe):')
for ticker, w in zip(tickers, opt_w):
    print(f'  {ticker}: {w:.3f} ({w*100:.1f}%)')

## 3. Weight Evolution Heatmap

Rolling monthly rebalance using momentum-based weights.

In [ ]:
lookback = 63
dates_list = close_wide.index.tolist()
weight_records = []

for i in range(lookback, len(dates_list), 21):
    d = dates_list[i]
    past_prices = close_wide.iloc[i - lookback]
    curr_prices = close_wide.iloc[i]
    ret = (curr_prices / past_prices - 1).fillna(0)
    ranks = ret.rank()
    w = ranks / ranks.sum()
    rec = {'date': d}
    rec.update(w.to_dict())
    weight_records.append(rec)

weights_over_time = pd.DataFrame(weight_records).set_index('date')

fig, ax = plt.subplots(figsize=(14, 5))
xticklabels = [str(d.date()) if hasattr(d, 'date') else str(d) for d in weights_over_time.index[::6]]
sns.heatmap(
    weights_over_time[tickers].T,
    ax=ax, cmap='YlOrRd', linewidths=0.5,
    xticklabels=False,
    yticklabels=tickers,
)
ax.set_title('Weight Evolution Heatmap (Monthly Rebalance, Momentum-Based)')
ax.set_xlabel('Date')
ax.set_ylabel('Ticker')
plt.tight_layout()
plt.show()

## 4. Portfolio Turnover Over Time

In [ ]:
w_arr = weights_over_time[tickers].values
turnover = np.abs(np.diff(w_arr, axis=0)).sum(axis=1) / 2

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(turnover)), turnover, alpha=0.7, color='steelblue')
ax.axhline(turnover.mean(), color='red', linestyle='--', label=f'Mean={turnover.mean():.3f}')
ax.set_xlabel('Rebalance Period')
ax.set_ylabel('One-Way Turnover')
ax.set_title('Portfolio Turnover Over Time')
ax.legend()
plt.tight_layout()
plt.show()

print(f'\nMean turnover: {turnover.mean():.3f} ({turnover.mean()*100:.1f}%)')
print(f'Max turnover:  {turnover.max():.3f} ({turnover.max()*100:.1f}%)')